# 01 - Explore the FIRMS data (MODIS and VIIRS)

This notebook only **describes** the raw data. It does not filter, convert confidence, or harmonize anything; those decisions belong to the team (see docs/METHOD.md).

Questions answered here:
1. Which columns does each sensor have, and how is confidence expressed?
2. How many detections per year and per sensor?
3. Are there date gaps?
4. How many points fall outside Colombia?
5. What visible differences are there between MODIS and VIIRS in the years where both exist?

*AI note: this notebook was written with the help of an AI assistant (Claude Code).*

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"

# Colombia approximate box (west, south, east, north), same as COLOMBIA_BBOX in scripts/download_firms.py
W, S, E, N = -79.5, -4.5, -66.5, 12.8

# Optional: DANE boundary file (data/processed/dane_mgn2018_municipios.geojson). If found, it is used for the outside-Colombia check.
DANE_GLOB = ["dane/*.shp", "dane/*.gpkg", "dane/*.geojson"]

## Load the CSV files
Every CSV under `data/raw` is loaded (archive downloads and files made by `scripts/download_firms.py`). The per-chunk cache folder `firms_cache` is skipped because the combined files already contain it.

In [ ]:
files = sorted(p for p in RAW.rglob("*.csv") if "firms_cache" not in p.parts)
if not files:
    raise SystemExit(f"No CSV files found in {RAW}. Run scripts/download_firms.py or add the archive download first.")

frames = []
for p in files:
    df = pd.read_csv(p, low_memory=False)
    df["source_file"] = p.name
    frames.append(df)
    print(f"{p.relative_to(RAW)}: {len(df):,} rows")

data = pd.concat(frames, ignore_index=True)
data["acq_date"] = pd.to_datetime(data["acq_date"], errors="coerce")
data["year"] = data["acq_date"].dt.year

# Sensor label built only from what the files say (instrument + satellite), no mapping applied.
inst = data["instrument"].astype(str) if "instrument" in data else "?"
sat = data["satellite"].astype(str) if "satellite" in data else "?"
data["sensor"] = inst + " / " + sat

print(f"
Total rows: {len(data):,}")
print("Unparseable dates:", int(data["acq_date"].isna().sum()))
dups = data.duplicated(subset=["latitude", "longitude", "acq_date", "acq_time", "sensor"]).sum()
print("Possible duplicate rows (same position, date, time, sensor):", int(dups))

## 1. Columns and confidence per sensor

In [ ]:
for sensor, g in data.groupby("sensor"):
    cols = [c for c in g.columns if g[c].notna().any() and c not in ("year", "sensor", "source_file")]
    print(f"{sensor}  ({len(g):,} rows)")
    print("  columns:", ", ".join(cols))
    print()

In [ ]:
# How confidence is expressed: look at the raw values, per sensor.
for sensor, g in data.groupby("sensor"):
    conf = g["confidence"]
    print(f"--- {sensor} | dtype: {conf.dtype}")
    if conf.nunique() <= 10:
        print(conf.value_counts(dropna=False).to_string())
    else:
        print(conf.describe().to_string())
    print()

## 2. Detections per year and sensor

In [ ]:
per_year = data.pivot_table(index="year", columns="sensor", values="latitude", aggfunc="size", fill_value=0)
display(per_year)
per_year.plot(kind="bar", figsize=(12, 4), title="Detections per year and sensor (raw counts)")
plt.ylabel("detections"); plt.tight_layout(); plt.show()

## 3. Date gaps
For each sensor: days between its first and last date with **no** detection at all. In a country as big as Colombia a fully empty day is unusual, so long runs are worth checking (missing download chunk or a real satellite outage). Short runs may be real.

In [ ]:
def gap_runs(dates):
    days = pd.DatetimeIndex(sorted(dates.dropna().unique()))
    full = pd.date_range(days.min(), days.max(), freq="D")
    missing = full.difference(days)
    runs = []
    for d in missing:
        if runs and (d - runs[-1][1]).days == 1:
            runs[-1][1] = d
        else:
            runs.append([d, d])
    return full, runs

for sensor, g in data.groupby("sensor"):
    full, runs = gap_runs(g["acq_date"])
    missing_days = sum((b - a).days + 1 for a, b in runs)
    print(f"{sensor}: {g['acq_date'].min().date()} -> {g['acq_date'].max().date()}, "
          f"{missing_days} empty days in {len(runs)} runs")
    longest = sorted(runs, key=lambda r: (r[1] - r[0]).days, reverse=True)[:5]
    for a, b in longest:
        print(f"    {a.date()} -> {b.date()} ({(b - a).days + 1} days)")

In [ ]:
monthly = data.groupby([data["acq_date"].dt.to_period("M"), "sensor"]).size().unstack(fill_value=0)
monthly.index = monthly.index.to_timestamp()
ax = monthly.plot(figsize=(12, 4), title="Detections per month and sensor (raw counts)")
ax.set_ylabel("detections"); plt.tight_layout(); plt.show()

## 4. Points outside Colombia

In [ ]:
inside_box = data["longitude"].between(W, E) & data["latitude"].between(S, N)
print(f"Outside the approximate box: {(~inside_box).sum():,} of {len(data):,}")

import glob
dane = [f for pat in DANE_GLOB for f in glob.glob(str(RAW / pat))]
if dane:
    import geopandas as gpd
    country = gpd.read_file(dane[0]).to_crs(4326).dissolve()
    pts = gpd.GeoDataFrame(data[["sensor"]], geometry=gpd.points_from_xy(data["longitude"], data["latitude"]), crs=4326)
    inside = pts.within(country.geometry.iloc[0])
    print(f"Outside the DANE boundary ({Path(dane[0]).name}): {(~inside).sum():,} of {len(data):,}")
    outside = data[~inside.values]
else:
    print("No DANE boundary found in data/raw/dane yet; using only the box. "
          "The box also covers parts of Venezuela, Brazil, Peru, Ecuador and Panama, so expect many foreign points inside it.")
    outside = data[~inside_box]

outside.groupby("sensor").size().rename("outside").to_frame()

## 5. MODIS versus VIIRS where both exist
Raw detection counts only (no filtering, no cells). This shows the size of the gap that the method in `docs/METHOD.md` has to deal with.

In [ ]:
data["family"] = data["instrument"].astype(str).where(data["instrument"].astype(str) == "MODIS", "VIIRS " + data["satellite"].astype(str))
fam_year = data.pivot_table(index="year", columns="family", values="latitude", aggfunc="size", fill_value=0)
display(fam_year.loc[2012:])

In [ ]:
viirs_cols = ["VIIRS SNPP"]  # S-NPP is the reference sensor (see METHOD.md)
modis_cols = [c for c in fam_year.columns if c.startswith("MODIS")]
both = fam_year.loc[2012:2025]
ratio = both[viirs_cols[0]] / both[modis_cols].sum(axis=1)
print(f"Ratio {viirs_cols[0]} / MODIS (all MODIS satellites), 2012-2025:")
display(ratio.round(2).to_frame("ratio"))
ratio.plot(figsize=(8, 3), marker="o", title=f"{viirs_cols[0]} detections per MODIS detection, by year")
plt.ylim(0, None); plt.tight_layout(); plt.show()

*Reading the result:* the ratio is stable across years, around 4 to 4.5. The method treats this as a property of the instruments (VIIRS pixels are about 7 times smaller), not as a change in fire activity. See section 3 of `docs/METHOD.md`.